# Day2 배터리 수명 Regression

분석: [model_report.md](../model_report.md)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler

import skala_ds_mini_project
from skala_ds_mini_project.features import (
    FEATURE_COLUMNS,
    build_feature_table,
    load_feature_inputs,
)
from skala_ds_mini_project.preprocessing import estimate_nominal_capacity

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "pyproject.toml").is_file(), "프로젝트 루트에서 실행하세요."
assert Path(skala_ds_mini_project.__file__).resolve().is_relative_to(PROJECT_ROOT), (
    "프로젝트 .venv 커널을 선택하세요: 다른 checkout이 import되었습니다."
)
DATA_DIR = PROJECT_ROOT / "data"
BATCH1_FILE = "2017-05-12_batchdata_updated_struct_errorcorrect.mat"
RANDOM_STATE = 42
VALID_SIZE = 0.20
N_SPLITS = 5
PAPER_TARGET_MAPE = 9.1
ID_COLUMNS = ["batch_id", "cell_id"]
FEATURE_ORDER = [
    "log_dQ_var",
    "dQ_kurt",
    "chargetime_med",
    "QD_max_minus_init",
    "Tmin_med",
    "early_fade",
]
assert set(FEATURE_ORDER) <= set(FEATURE_COLUMNS)
# 앞에서부터 하나씩 추가하여 F1~F6을 만든다.
feature_sets = {}
for size in range(1, len(FEATURE_ORDER) + 1):
    feature_sets[f"F{size}"] = FEATURE_ORDER[:size]
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

# 고정 설정만 사용하며 CV/Valid/Test 결과에 따른 튜닝은 하지 않는다.
MODELS = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, n_jobs=1, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=RANDOM_STATE
    ),
}

## MAPE와 학습

In [2]:
def cycle_mape(y_true_log, y_pred_log):
    """로그 실제값과 예측값을 cycle 수로 복원하여 MAPE(%)를 계산한다."""
    true_cycle = 10.0 ** np.asarray(y_true_log, dtype=float)
    pred_cycle = 10.0 ** np.asarray(y_pred_log, dtype=float)
    return float(np.mean(np.abs((true_cycle - pred_cycle) / true_cycle)) * 100)


def fit_model(model_name, train_features, train_target):
    """학습 데이터로 결측값 처리·표준화·회귀를 순서대로 학습한다."""
    # fit은 통계를 학습한다. 평가 데이터는 중앙값과 평균 계산에 넣지 않는다.
    imputer = SimpleImputer(strategy="median")
    train_imputed = imputer.fit_transform(train_features)
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_imputed)

    # fold마다 새 모델을 학습한다. 트리도 기존 결과 보존을 위해 표준화한다.
    model = clone(MODELS[model_name])
    model.fit(train_scaled, train_target)
    return imputer, scaler, model


def predict_model(fitted_model, evaluation_features):
    """학습 때 구한 통계만 적용하여 평가 데이터의 로그 수명을 예측한다."""
    imputer, scaler, model = fitted_model
    # transform은 통계를 다시 학습하지 않는다. Valid/Test 정보 누수를 막는다.
    evaluation_imputed = imputer.transform(evaluation_features)
    evaluation_scaled = scaler.transform(evaluation_imputed)
    return model.predict(evaluation_scaled)

## Batch 1 분할

In [3]:
def eligible_cell_ids(summary, dq):
    """양의 유효 수명과 ΔQ 데이터가 있는 cell만 사용한다."""
    assert summary["cycle"].le(100).all()
    assert not summary.duplicated([*ID_COLUMNS, "cycle"]).any(), "중복 summary cycle"
    cell_labels = summary[[*ID_COLUMNS, "cycle_life"]].drop_duplicates()
    summary_labels = cell_labels.set_index(ID_COLUMNS)["cycle_life"]
    assert summary_labels.index.is_unique, "cell별 summary label 불일치"
    dq_labels = dq.set_index(ID_COLUMNS)["cycle_life"]
    assert dq_labels.index.is_unique, "중복 ΔQ cell"
    labels = dq_labels.loc[np.isfinite(dq_labels) & dq_labels.gt(0)].sort_index()
    assert len(labels) > 0, "eligible cell이 없습니다."
    assert labels.index.isin(summary_labels.index).all(), "초기 summary가 없는 eligible cell"
    assert np.array_equal(labels.to_numpy(), summary_labels.loc[labels.index].to_numpy()), (
        "summary와 ΔQ label 불일치"
    )
    return labels.index


def rows_for_cells(frame, cell_ids):
    """(batch_id, cell_id) 쌍이 일치하는 행을 선택한다."""
    # cell_id가 다른 batch에서 반복돼도 MultiIndex의 두 ID 쌍으로 구분한다.
    keys = pd.MultiIndex.from_frame(frame[ID_COLUMNS])
    return frame.loc[keys.isin(cell_ids)].copy()


def feature_table_for_cells(summary, dq, cell_ids, nominal):
    """학습 cell에서 추정한 nominal로 기존 feature 추출식을 적용한다."""
    assert np.isfinite(nominal) and nominal > 0
    assert summary["cycle"].le(100).all()
    table = build_feature_table(
        rows_for_cells(summary, cell_ids),
        rows_for_cells(dq, cell_ids),
        nominal=nominal,
    ).set_index(ID_COLUMNS)
    assert table.index.is_unique, "중복 feature cell"
    assert table.index.sort_values().equals(cell_ids.sort_values()), (
        "feature 추출 후 cell 누락/추가: 모집단을 조용히 변경하지 않습니다."
    )
    # split 당시 ID 순서로 맞춰 입력과 정답의 행 순서를 보존한다.
    table = table.loc[cell_ids]
    assert np.isfinite(table["log_cycle_life"]).all()
    assert np.allclose(table["log_cycle_life"], np.log10(table["cycle_life"]))
    assert not np.isinf(table[FEATURE_ORDER].to_numpy()).any()
    return table

In [4]:
# 기본 loader는 모든 batch를 읽으므로 파일을 반드시 명시한다.
b1_summary, b1_dq, _, _ = load_feature_inputs(DATA_DIR, batch_files=(BATCH1_FILE,))
b1_summary = b1_summary.loc[b1_summary["cycle"].le(100)].copy()
assert b1_summary["batch_id"].eq(1).all() and b1_dq["batch_id"].eq(1).all()
b1_ids = eligible_cell_ids(b1_summary, b1_dq)
train_positions, valid_positions = train_test_split(
    np.arange(len(b1_ids)),
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)
train_ids = b1_ids.take(train_positions)
valid_ids = b1_ids.take(valid_positions)
assert train_ids.intersection(valid_ids).empty
assert len(train_ids) + len(valid_ids) == len(b1_ids)
# Hold-out을 제외한 Train 안에서만 5개 fold를 나눈다.
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

## CV 평가

In [5]:
fold_cache = []
fold_nominals = []
for fold_train_positions, fold_eval_positions in cv.split(train_ids):
    fold_train_ids = train_ids.take(fold_train_positions)
    fold_eval_ids = train_ids.take(fold_eval_positions)
    assert fold_train_ids.intersection(fold_eval_ids).empty
    assert fold_train_ids.intersection(valid_ids).empty
    assert fold_eval_ids.intersection(valid_ids).empty
    # nominal도 fold 학습 부분에서만 추정하고 평가 부분에는 같은 값을 적용한다.
    fold_nominal = estimate_nominal_capacity(rows_for_cells(b1_summary, fold_train_ids))
    fold_train_table = feature_table_for_cells(b1_summary, b1_dq, fold_train_ids, fold_nominal)
    fold_eval_table = feature_table_for_cells(b1_summary, b1_dq, fold_eval_ids, fold_nominal)
    fold_cache.append((fold_train_table, fold_eval_table))
    fold_nominals.append(fold_nominal)
assert len(fold_cache) == N_SPLITS


def cv_mape(model_name, features):
    """같은 Train fold들을 평가하고 fold MAPE의 단순 평균을 반환한다."""
    assert features == FEATURE_ORDER[: len(features)]
    fold_scores = []
    for fold_train, fold_eval in fold_cache:
        assert fold_train[features].notna().any().all(), "중앙값을 구할 수 없는 학습 열"
        # 각 fold의 학습 부분에서만 fit하고 평가 부분에는 transform/predict만 한다.
        fitted_model = fit_model(model_name, fold_train[features], fold_train["log_cycle_life"])
        prediction = predict_model(fitted_model, fold_eval[features])
        fold_score = cycle_mape(fold_eval["log_cycle_life"], prediction)
        fold_scores.append(fold_score)
    assert np.isfinite(fold_scores).all()
    return float(np.mean(fold_scores))

## Feature 비교

In [6]:
comparison_records = []
for feature_set, features in feature_sets.items():
    for model_name in MODELS:
        if model_name == "Linear Regression" and feature_set != "F1":
            continue
        comparison_records.append(
            {
                "Model": model_name,
                "Feature Set": feature_set,
                "Features": ", ".join(features),
                "Batch 1 CV MAPE (%)": cv_mape(model_name, features),
            }
        )
model_comparison = pd.DataFrame(comparison_records)
assert len(model_comparison) == 1 + 3 * len(feature_sets)
assert not model_comparison.duplicated(["Model", "Feature Set"]).any()

# 비교 표의 최솟값을 고르는 것이 아니라 Ridge + F3를 명시적으로 고정한다.
selected_model = "Ridge"
ridge_rows = model_comparison.loc[model_comparison["Model"].eq(selected_model)]
ridge_comparison = ridge_rows.set_index("Feature Set")
selected_feature_set = "F3"
selected_features = feature_sets[selected_feature_set].copy()
train_cv_mape = float(ridge_comparison.loc[selected_feature_set, "Batch 1 CV MAPE (%)"])
assert selected_features == FEATURE_ORDER[: len(selected_features)]
selection_frozen = True

## Hold-out 평가

In [7]:
assert selection_frozen
train_nominal = estimate_nominal_capacity(rows_for_cells(b1_summary, train_ids))
train_table = feature_table_for_cells(b1_summary, b1_dq, train_ids, train_nominal)
valid_table = feature_table_for_cells(b1_summary, b1_dq, valid_ids, train_nominal)
assert train_table[selected_features].notna().any().all()
# Train 전체에서 한 번 학습한다. Hold-out을 합쳐 재학습하지 않는다.
frozen_model = fit_model(
    selected_model, train_table[selected_features], train_table["log_cycle_life"]
)
valid_prediction = predict_model(frozen_model, valid_table[selected_features])
valid_mape = cycle_mape(valid_table["log_cycle_life"], valid_prediction)
assert np.isfinite(valid_mape)
holdout_assessed = True

## 최종 Test

In [8]:
assert selection_frozen and holdout_assessed
if globals().get("final_test_started", False):
    raise RuntimeError("이 커널에서 최종 Batch 2 Test는 이미 시작되었습니다.")
final_test_started = True

b2_summary, b2_dq, _, _ = load_feature_inputs(
    DATA_DIR,
    batch_files=("2018-02-20_batchdata_updated_struct_errorcorrect.mat",),
)
b2_summary = b2_summary.loc[b2_summary["cycle"].le(100)].assign(batch_id=2)
b2_dq = b2_dq.assign(batch_id=2)
test_ids = eligible_cell_ids(b2_summary, b2_dq)
assert test_ids.intersection(b1_ids).empty
test_table = feature_table_for_cells(b2_summary, b2_dq, test_ids, train_nominal)
# Hold-out 때 학습한 결측값 처리기·스케일러·모델을 그대로 사용한다.
# Test에서는 transform과 predict만 호출한다.
test_prediction = predict_model(frozen_model, test_table[selected_features])
test_mape = cycle_mape(test_table["log_cycle_life"], test_prediction)
assert np.isfinite(test_mape)

In [9]:
display(Markdown("## 1. Dataset"))
dataset = pd.DataFrame(
    [
        ["Train (Batch 1)", len(train_ids), "log10(cycle_life)"],
        ["Valid (Batch 1 Hold-out)", len(valid_ids), "log10(cycle_life)"],
        ["Test (Batch 2)", len(test_ids), "log10(cycle_life)"],
    ],
    columns=["Dataset", "Cells", "Target"],
)
display(dataset)

display(Markdown("## 2. Feature Sets"))
feature_set_records = []
for name, features in feature_sets.items():
    feature_set_records.append({"Feature Set": name, "Features": ", ".join(features)})
feature_sets_table = pd.DataFrame(feature_set_records)
display(feature_sets_table)

## 1. Dataset

,Dataset,Cells,Target
0,Train (Batch 1),36,log10(cycle_life)
1,Valid (Batch 1 Hold-out),10,log10(cycle_life)
2,Test (Batch 2),39,log10(cycle_life)


## 2. Feature Sets

,Feature Set,Features
0,F1,log_dQ_var
1,F2,"log_dQ_var, dQ_kurt"
2,F3,"log_dQ_var, dQ_kurt, chargetime_med"
3,F4,"log_dQ_var, dQ_kurt, chargetime_med, QD_max_minus_init"
4,F5,"log_dQ_var, dQ_kurt, chargetime_med, QD_max_minus_init, Tmin_med"
5,F6,"log_dQ_var, dQ_kurt, chargetime_med, QD_max_minus_init, Tmin_med, early_fade"


In [10]:
display(Markdown("## 3. Model Comparison — Batch 1 CV MAPE (%)"))
comparison_rows = model_comparison.loc[~model_comparison["Model"].eq("Linear Regression")]
comparison_table = comparison_rows.pivot(
    index="Feature Set", columns="Model", values="Batch 1 CV MAPE (%)"
)
comparison_table = comparison_table.reindex(
    index=list(feature_sets), columns=["Ridge", "Random Forest", "Gradient Boosting"]
)
comparison_table = comparison_table.rename_axis(columns=None).reset_index()
baseline_table = model_comparison.loc[model_comparison["Model"].eq("Linear Regression")].copy()
display(comparison_table)
display(baseline_table)

display(Markdown("## 4. Selected Model"))
selected_model_table = pd.DataFrame(
    [
        {
            "Model": selected_model,
            "Feature Set": selected_feature_set,
            "Features": ", ".join(selected_features),
            "Batch 1 CV MAPE (%)": train_cv_mape,
        }
    ]
)
display(selected_model_table)

## 3. Model Comparison — Batch 1 CV MAPE (%)

,Feature Set,Ridge,Random Forest,Gradient Boosting
0,F1,10.063,9.627,10.584
1,F2,10.656,10.484,11.340
2,F3,10.416,10.215,10.952
3,F4,10.770,10.461,11.007
4,F5,10.729,10.186,11.487
5,F6,11.778,10.323,11.424


,Model,Feature Set,Features,Batch 1 CV MAPE (%)
0,Linear Regression,F1,log_dQ_var,10.059


## 4. Selected Model

,Model,Feature Set,Features,Batch 1 CV MAPE (%)
0,Ridge,F3,"log_dQ_var, dQ_kurt, chargetime_med",10.416


In [11]:
gap_train_valid = valid_mape - train_cv_mape
gap_valid_test = test_mape - valid_mape
gap_target_test = test_mape - PAPER_TARGET_MAPE
performance = pd.DataFrame(
    [
        ["Train (Batch 1 CV)", train_cv_mape, "Batch 1 CV 평균"],
        ["Valid (Batch 1 Hold-out)", valid_mape, "Batch 1 Hold-out"],
        ["Test (Batch 2)", test_mape, "Batch 2 Test"],
        ["Gap (Train-Valid)", gap_train_valid, "pp; Valid \u2212 CV"],
        ["Gap (Valid-Test)", gap_valid_test, "pp; Test \u2212 Valid"],
        ["Gap (Target-Test)", gap_target_test, "pp; Test \u2212 9.1%"],
    ],
    columns=["구분", "MAPE (%)", "비고"],
)
assert performance.shape == (6, 3)
display(Markdown("## 5. Performance"))
display(performance)

## 5. Performance

,구분,MAPE (%),비고
0,Train (Batch 1 CV),10.416,Batch 1 CV 평균
1,Valid (Batch 1 Hold-out),8.344,Batch 1 Hold-out
2,Test (Batch 2),37.345,Batch 2 Test
3,Gap (Train-Valid),-2.072,pp; Valid − CV
4,Gap (Valid-Test),29.000,pp; Test − Valid
5,Gap (Target-Test),28.245,pp; Test − 9.1%


# 결과 해석

In [16]:
# Batch 2 sample별 예측 오차 확인
test_actual_cycle = 10.0 ** test_table["log_cycle_life"].to_numpy()
test_pred_cycle = 10.0 ** np.asarray(test_prediction)

test_diagnostics = pd.DataFrame(
    {
        "Actual Cycle": test_actual_cycle,
        "Predicted Cycle": test_pred_cycle,
    },
    index=test_table.index,
)

test_diagnostics["APE (%)"] = (
    np.abs(
        test_diagnostics["Actual Cycle"]
        - test_diagnostics["Predicted Cycle"]
    )
    / test_diagnostics["Actual Cycle"]
    * 100
)

display(Markdown("### Batch 2 APE 요약"))

ape_summary = pd.DataFrame(
    {
        "Value": [
            test_diagnostics["APE (%)"].mean(),
            test_diagnostics["APE (%)"].median(),
            test_diagnostics["APE (%)"].quantile(0.75),
            test_diagnostics["APE (%)"].quantile(0.90),
            test_diagnostics["APE (%)"].max(),
            (test_diagnostics["APE (%)"] > 20).sum(),
            (test_diagnostics["APE (%)"] > 50).sum(),
        ]
    },
    index=[
        "Mean APE (%)",
        "Median APE (%)",
        "75% Quantile (%)",
        "90% Quantile (%)",
        "Max APE (%)",
        "Cells > 20% APE",
        "Cells > 50% APE",
    ],
)

display(ape_summary)

display(Markdown("### APE가 큰 Batch 2 Cell Top 10"))

display(
    test_diagnostics
    .sort_values("APE (%)", ascending=False)
    .head(10)
)

### Batch 2 APE 요약

,Value
Mean APE (%),37.345
Median APE (%),37.475
75% Quantile (%),47.430
90% Quantile (%),58.145
Max APE (%),74.428
Cells > 20% APE,31.000
Cells > 50% APE,10.000


### APE가 큰 Batch 2 Cell Top 10

Actual Cycle  Predicted Cycle  APE (%)
batch_id cell_id                                        
2        6             393.000          685.500   74.428
         15            396.000          684.509   72.856
         18            449.000          739.984   64.807
         19            392.000          623.918   59.163
         2             424.000          669.455   57.890
         21            408.000          632.217   54.955
         28            437.000          659.403   50.893
         29            452.000          681.455   50.764
         11            449.000          673.618   50.026
         31            425.000          637.599   50.023

In [17]:
# Batch 2 예측 방향(Bias) 확인

bias_df = test_diagnostics.copy()

# 양수: 과대예측 / 음수: 과소예측
bias_df["Signed Error (%)"] = (
    (
        bias_df["Predicted Cycle"]
        - bias_df["Actual Cycle"]
    )
    / bias_df["Actual Cycle"]
    * 100
)

# 예측값이 실제값의 몇 배인지
bias_df["Pred / Actual"] = (
    bias_df["Predicted Cycle"]
    / bias_df["Actual Cycle"]
)

bias_summary = pd.DataFrame(
    {
        "Value": [
            bias_df["Signed Error (%)"].mean(),
            bias_df["Signed Error (%)"].median(),
            (bias_df["Signed Error (%)"] > 0).sum(),
            (bias_df["Signed Error (%)"] < 0).sum(),
            (bias_df["Signed Error (%)"] == 0).sum(),
            bias_df["Pred / Actual"].mean(),
            bias_df["Pred / Actual"].median(),
            len(bias_df),
        ]
    },
    index=[
        "Mean Signed Error (%)",
        "Median Signed Error (%)",
        "Over-predicted Cells",
        "Under-predicted Cells",
        "Exact Cells",
        "Mean Pred / Actual",
        "Median Pred / Actual",
        "Total Cells",
    ],
)

display(Markdown("### Batch 2 Prediction Bias"))
display(bias_summary)

# 기존 Test MAPE와 현재 재계산 MAPE도 함께 확인
print(f"Recomputed MAPE : {bias_df['APE (%)'].mean():.3f}%")
print(f"Existing test_mape : {test_mape}")

### Batch 2 Prediction Bias

,Value
Mean Signed Error (%),33.893
Median Signed Error (%),37.475
Over-predicted Cells,34.000
Under-predicted Cells,5.000
Exact Cells,0.000
Mean Pred / Actual,1.339
Median Pred / Actual,1.375
Total Cells,39.000


Recomputed MAPE : 37.345%
Existing test_mape : 37.34452704441607


In [20]:
# Batch 1 Hold-out과 Batch 2의 Prediction Bias 비교

def prediction_bias_summary(table, prediction, name):
    actual = 10.0 ** table["log_cycle_life"].to_numpy()
    predicted = 10.0 ** np.asarray(prediction)

    signed_error = (predicted - actual) / actual * 100
    ratio = predicted / actual

    return {
        "Dataset": name,
        "Cells": len(actual),
        "MAPE (%)": np.mean(np.abs(signed_error)),
        "Mean Signed Error (%)": np.mean(signed_error),
        "Median Signed Error (%)": np.median(signed_error),
        "Over-predicted Cells": np.sum(signed_error > 0),
        "Under-predicted Cells": np.sum(signed_error < 0),
        "Mean Pred / Actual": np.mean(ratio),
        "Median Pred / Actual": np.median(ratio),
    }


bias_comparison = pd.DataFrame([
    prediction_bias_summary(
        valid_table,
        valid_prediction,
        "Valid (Batch 1 Hold-out)"
    ),
    prediction_bias_summary(
        test_table,
        test_prediction,
        "Test (Batch 2)"
    ),
])

display(Markdown("### Valid vs Test Prediction Bias"))
display(bias_comparison.set_index("Dataset"))

### Valid vs Test Prediction Bias

,Cells,MAPE (%),Mean Signed Error (%),Median Signed Error (%),Over-predicted Cells,Under-predicted Cells,Mean Pred / Actual,Median Pred / Actual
Dataset,,,,,,,,
Valid (Batch 1 Hold-out),10,8.344,-2.529,-2.838,4,6,0.975,0.972
Test (Batch 2),39,37.345,33.893,37.475,34,5,1.339,1.375


In [22]:
# Batch 1 Train / Valid / Batch 2 Test의 feature 분포 비교
# frozen_model에서 실제 학습된 imputer/scaler를 그대로 사용한다.

imputer, scaler, ridge_model = frozen_model

def get_scaled_features(table):
    imputed = imputer.transform(table[selected_features])
    scaled = scaler.transform(imputed)

    return pd.DataFrame(
        scaled,
        columns=selected_features,
        index=table.index,
    )


train_scaled = get_scaled_features(train_table)
valid_scaled = get_scaled_features(valid_table)
test_scaled = get_scaled_features(test_table)


shift_records = []

for feature in selected_features:
    train_min = train_scaled[feature].min()
    train_max = train_scaled[feature].max()

    test_outside = (
        (test_scaled[feature] < train_min)
        | (test_scaled[feature] > train_max)
    )

    shift_records.append(
        {
            "Feature": feature,

            "Train Mean (z)": train_scaled[feature].mean(),
            "Valid Mean (z)": valid_scaled[feature].mean(),
            "Test Mean (z)": test_scaled[feature].mean(),

            "Train Median (z)": train_scaled[feature].median(),
            "Valid Median (z)": valid_scaled[feature].median(),
            "Test Median (z)": test_scaled[feature].median(),

            "Train Min (z)": train_min,
            "Train Max (z)": train_max,

            "Test Min (z)": test_scaled[feature].min(),
            "Test Max (z)": test_scaled[feature].max(),

            "Test Outside Train Range": test_outside.sum(),
            "Test Outside Train Range (%)": test_outside.mean() * 100,
        }
    )


feature_shift = pd.DataFrame(shift_records).set_index("Feature")

display(Markdown("### Batch 1 → Batch 2 Feature Shift"))
display(feature_shift.transpose())

### Batch 1 → Batch 2 Feature Shift

Feature,log_dQ_var,dQ_kurt,chargetime_med
Train Mean (z),-0.000,0.000,-0.000
Valid Mean (z),-0.195,-0.142,0.315
Test Mean (z),0.828,0.127,-0.738
Train Median (z),0.162,-0.171,-0.074
Valid Median (z),-0.285,-0.221,0.179
Test Median (z),1.118,-0.012,-0.714
Train Min (z),-2.945,-0.548,-1.853
Train Max (z),1.482,5.721,2.341
Test Min (z),-1.439,-0.910,-0.846
Test Max (z),2.185,1.611,-0.649


In [24]:
# Ridge feature별 예측 기여도 분해
# 예측 대상은 log_cycle_life이므로 contribution도 log10 scale이다.

ridge_coef = np.ravel(ridge_model.coef_)

contribution_records = []

for feature, coef in zip(selected_features, ridge_coef):
    train_contrib = train_scaled[feature] * coef
    valid_contrib = valid_scaled[feature] * coef
    test_contrib = test_scaled[feature] * coef

    contribution_records.append({
        "Feature": feature,
        "Ridge Coef": coef,

        "Train Mean z": train_scaled[feature].mean(),
        "Valid Mean z": valid_scaled[feature].mean(),
        "Test Mean z": test_scaled[feature].mean(),

        "Train Mean Contribution": train_contrib.mean(),
        "Valid Mean Contribution": valid_contrib.mean(),
        "Test Mean Contribution": test_contrib.mean(),

        # 양수면 해당 feature shift가
        # Valid보다 Test 예측 log_cycle_life를 높이는 방향
        "Test - Valid Contribution": (
            test_contrib.mean() - valid_contrib.mean()
        ),
    })

contribution_df = (
    pd.DataFrame(contribution_records)
    .set_index("Feature")
)

display(Markdown("### Ridge Feature Contribution Shift"))
display(contribution_df.transpose())


# 전체 feature contribution 변화와
# 실제 prediction 평균 변화가 일치하는지도 확인
total_contribution_shift = (
    contribution_df["Test - Valid Contribution"].sum()
)

observed_prediction_shift = (
    np.mean(test_prediction)
    - np.mean(valid_prediction)
)

print(
    f"Sum of feature contribution shift (Test - Valid): "
    f"{total_contribution_shift:.4f}"
)

print(
    f"Observed mean prediction shift (Test - Valid): "
    f"{observed_prediction_shift:.4f}"
)

print(
    f"Ridge intercept: {ridge_model.intercept_:.4f}"
)

### Ridge Feature Contribution Shift

Feature,log_dQ_var,dQ_kurt,chargetime_med
Ridge Coef,-0.066,-0.003,0.012
Train Mean z,-0.000,0.000,-0.000
Valid Mean z,-0.195,-0.142,0.315
Test Mean z,0.828,0.127,-0.738
Train Mean Contribution,0.000,-0.000,-0.000
Valid Mean Contribution,0.013,0.000,0.004
Test Mean Contribution,-0.054,-0.000,-0.009
Test - Valid Contribution,-0.067,-0.001,-0.013


Sum of feature contribution shift (Test - Valid): -0.0804
Observed mean prediction shift (Test - Valid): -0.0804
Ridge intercept: 2.9102


In [26]:
# Batch 1 Hold-out과 Batch 2의
# 실제 target 분포 변화와 prediction 분포 변화를 비교

def target_prediction_summary(table, prediction, name):
    actual_log = table["log_cycle_life"].to_numpy()
    pred_log = np.asarray(prediction)

    actual_cycle = 10.0 ** actual_log
    pred_cycle = 10.0 ** pred_log

    return {
        "Dataset": name,

        "Actual Mean (log10)": actual_log.mean(),
        "Actual Median (log10)": np.median(actual_log),

        "Pred Mean (log10)": pred_log.mean(),
        "Pred Median (log10)": np.median(pred_log),

        "Actual Mean (cycle)": actual_cycle.mean(),
        "Actual Median (cycle)": np.median(actual_cycle),

        "Pred Mean (cycle)": pred_cycle.mean(),
        "Pred Median (cycle)": np.median(pred_cycle),
    }


target_shift = pd.DataFrame([
    target_prediction_summary(
        valid_table,
        valid_prediction,
        "Valid (Batch 1 Hold-out)"
    ),
    target_prediction_summary(
        test_table,
        test_prediction,
        "Test (Batch 2)"
    ),
]).set_index("Dataset")

display(Markdown("### Actual Target vs Prediction Shift"))
display(target_shift.transpose())


# Valid -> Test 변화량
actual_log_shift = (
    target_shift.loc["Test (Batch 2)", "Actual Mean (log10)"]
    - target_shift.loc["Valid (Batch 1 Hold-out)", "Actual Mean (log10)"]
)

pred_log_shift = (
    target_shift.loc["Test (Batch 2)", "Pred Mean (log10)"]
    - target_shift.loc["Valid (Batch 1 Hold-out)", "Pred Mean (log10)"]
)

print(f"Actual mean log10 shift (Test - Valid): {actual_log_shift:.4f}")
print(f"Predicted mean log10 shift (Test - Valid): {pred_log_shift:.4f}")

print()

print(
    "Actual cycle ratio (Test / Valid): "
    f"{10 ** actual_log_shift:.3f}"
)

print(
    "Predicted cycle ratio (Test / Valid): "
    f"{10 ** pred_log_shift:.3f}"
)

### Actual Target vs Prediction Shift

Dataset,Valid (Batch 1 Hold-out),Test (Batch 2)
Actual Mean (log10),2.940,2.727
Actual Median (log10),2.942,2.674
Pred Mean (log10),2.927,2.847
Pred Median (log10),2.937,2.829
Actual Mean (cycle),895.300,565.744
Actual Median (cycle),874.500,472.000
Pred Mean (cycle),856.629,711.160
Pred Median (cycle),865.271,674.153


Actual mean log10 shift (Test - Valid): -0.2128
Predicted mean log10 shift (Test - Valid): -0.0804

Actual cycle ratio (Test / Valid): 0.613
Predicted cycle ratio (Test / Valid): 0.831


In [27]:
# Batch 2에서 Ridge가 놓친 부분(residual)의 구조 확인
# residual < 0 이면 실제값보다 과대예측한 것

test_actual_log = test_table["log_cycle_life"].to_numpy()
test_pred_log = np.asarray(test_prediction)

test_residual_log = test_actual_log - test_pred_log

residual_summary = pd.DataFrame(
    {
        "Value": [
            test_residual_log.mean(),
            np.median(test_residual_log),
            test_residual_log.std(),
            test_residual_log.min(),
            test_residual_log.max(),
        ]
    },
    index=[
        "Mean Residual (log10)",
        "Median Residual (log10)",
        "Std Residual (log10)",
        "Min Residual (log10)",
        "Max Residual (log10)",
    ],
)

display(Markdown("### Batch 2 Residual Summary"))
display(residual_summary)


# residual이 특정 feature와 함께 변하는지 확인
residual_feature_records = []

for feature in selected_features:
    corr = np.corrcoef(
        test_scaled[feature].to_numpy(),
        test_residual_log
    )[0, 1]

    residual_feature_records.append(
        {
            "Feature": feature,
            "Correlation with Residual": corr,
        }
    )

residual_feature_corr = (
    pd.DataFrame(residual_feature_records)
    .set_index("Feature")
)

display(Markdown("### Feature vs Batch 2 Residual Correlation"))
display(residual_feature_corr)


# 참고: log residual을 배수 관점으로 해석
print(
    "Mean Actual / Predicted ratio:",
    f"{10 ** test_residual_log.mean():.3f}"
)

print(
    "Median Actual / Predicted ratio:",
    f"{10 ** np.median(test_residual_log):.3f}"
)

### Batch 2 Residual Summary

,Value
Mean Residual (log10),-0.119
Median Residual (log10),-0.138
Std Residual (log10),0.083
Min Residual (log10),-0.242
Max Residual (log10),0.103


### Feature vs Batch 2 Residual Correlation

,Correlation with Residual
Feature,
log_dQ_var,-0.739
dQ_kurt,0.713
chargetime_med,-0.812


Mean Actual / Predicted ratio: 0.760
Median Actual / Predicted ratio: 0.727


In [29]:
# Batch 1 Valid vs Batch 2 Test
# residual-feature correlation 비교

def residual_feature_analysis(table, prediction, scaled_df, name):
    actual_log = table["log_cycle_life"].to_numpy()
    pred_log = np.asarray(prediction)

    residual = actual_log - pred_log

    result = []

    for feature in selected_features:
        corr = np.corrcoef(
            scaled_df[feature].to_numpy(),
            residual
        )[0, 1]

        result.append({
            "Dataset": name,
            "Feature": feature,
            "Residual Correlation": corr,
        })

    return pd.DataFrame(result)


valid_residual_corr = residual_feature_analysis(
    valid_table,
    valid_prediction,
    valid_scaled,
    "Valid (Batch 1)"
)

test_residual_corr = residual_feature_analysis(
    test_table,
    test_prediction,
    test_scaled,
    "Test (Batch 2)"
)

residual_corr_comparison = (
    pd.concat(
        [valid_residual_corr, test_residual_corr],
        ignore_index=True
    )
    .pivot(
        index="Feature",
        columns="Dataset",
        values="Residual Correlation"
    )
)

residual_corr_comparison["Difference (Test - Valid)"] = (
    residual_corr_comparison["Test (Batch 2)"]
    - residual_corr_comparison["Valid (Batch 1)"]
)

display(Markdown("### Residual Correlation: Valid vs Test"))
display(residual_corr_comparison.transpose())


# residual 자체의 분포도 같이 비교
def residual_summary(table, prediction):
    actual_log = table["log_cycle_life"].to_numpy()
    pred_log = np.asarray(prediction)

    residual = actual_log - pred_log

    return {
        "Mean": residual.mean(),
        "Median": np.median(residual),
        "Std": residual.std(),
    }


residual_distribution = pd.DataFrame(
    {
        "Valid (Batch 1)": residual_summary(
            valid_table,
            valid_prediction
        ),
        "Test (Batch 2)": residual_summary(
            test_table,
            test_prediction
        ),
    }
)

display(Markdown("### Residual Distribution Comparison"))
display(residual_distribution)

### Residual Correlation: Valid vs Test

Feature,chargetime_med,dQ_kurt,log_dQ_var
Dataset,,,
Test (Batch 2),-0.812,0.713,-0.739
Valid (Batch 1),0.600,0.410,-0.662
Difference (Test - Valid),-1.412,0.302,-0.076


### Residual Distribution Comparison

,Valid (Batch 1),Test (Batch 2)
Mean,0.013,-0.119
Median,0.013,-0.138
Std,0.041,0.083


In [30]:
# Train / Valid / Test에서
# 각 feature와 실제 log_cycle_life의 관계 비교

def feature_target_correlations(
    scaled_df,
    table,
    dataset_name
):
    y = table["log_cycle_life"].to_numpy()

    records = []

    for feature in selected_features:
        x = scaled_df[feature].to_numpy()

        corr = np.corrcoef(x, y)[0, 1]

        # 단순 직선 기울기
        slope = np.polyfit(x, y, 1)[0]

        records.append({
            "Dataset": dataset_name,
            "Feature": feature,
            "Target Correlation": corr,
            "Simple Slope": slope,
        })

    return pd.DataFrame(records)


train_relation = feature_target_correlations(
    train_scaled,
    train_table,
    "Train (Batch 1)"
)

valid_relation = feature_target_correlations(
    valid_scaled,
    valid_table,
    "Valid (Batch 1)"
)

test_relation = feature_target_correlations(
    test_scaled,
    test_table,
    "Test (Batch 2)"
)


relation_df = pd.concat(
    [
        train_relation,
        valid_relation,
        test_relation,
    ],
    ignore_index=True
)


# Correlation 비교
corr_comparison = relation_df.pivot(
    index="Feature",
    columns="Dataset",
    values="Target Correlation",
)

corr_comparison["Ridge Coef"] = ridge_coef

display(Markdown("### Feature ↔ Actual Target Correlation"))
display(corr_comparison)


# 단순 회귀 기울기 비교
slope_comparison = relation_df.pivot(
    index="Feature",
    columns="Dataset",
    values="Simple Slope",
)

slope_comparison["Ridge Coef"] = ridge_coef

display(Markdown("### Feature ↔ Actual Target Simple Slope"))
display(slope_comparison)

### Feature ↔ Actual Target Correlation

Dataset,Test (Batch 2),Train (Batch 1),Valid (Batch 1),Ridge Coef
Feature,,,,
chargetime_med,-0.935,0.719,0.824,-0.066
dQ_kurt,0.653,0.303,0.373,-0.003
log_dQ_var,-0.918,-0.846,-0.951,0.012


### Feature ↔ Actual Target Simple Slope

Dataset,Test (Batch 2),Train (Batch 1),Valid (Batch 1),Ridge Coef
Feature,,,,
chargetime_med,-2.206,0.065,0.118,-0.066
dQ_kurt,0.138,0.027,0.142,-0.003
log_dQ_var,-0.123,-0.076,-0.100,0.012


In [31]:
# Ridge coefficient mapping을 feature 이름 기준으로 정확하게 생성
ridge_coef_map = pd.Series(
    ridge_coef,
    index=selected_features,
    name="Ridge Coef"
)

display(Markdown("### Correct Ridge Coefficient Mapping"))
display(ridge_coef_map.to_frame())


# Train / Valid / Test feature 간 correlation 비교
def feature_corr_pairs(scaled_df, dataset_name):
    corr = scaled_df[selected_features].corr()

    records = []

    for i in range(len(selected_features)):
        for j in range(i + 1, len(selected_features)):
            f1 = selected_features[i]
            f2 = selected_features[j]

            records.append({
                "Dataset": dataset_name,
                "Feature Pair": f"{f1} ↔ {f2}",
                "Correlation": corr.loc[f1, f2],
            })

    return pd.DataFrame(records)


train_feature_corr = feature_corr_pairs(
    train_scaled,
    "Train (Batch 1)"
)

valid_feature_corr = feature_corr_pairs(
    valid_scaled,
    "Valid (Batch 1)"
)

test_feature_corr = feature_corr_pairs(
    test_scaled,
    "Test (Batch 2)"
)


feature_corr_comparison = (
    pd.concat(
        [
            train_feature_corr,
            valid_feature_corr,
            test_feature_corr,
        ],
        ignore_index=True
    )
    .pivot(
        index="Feature Pair",
        columns="Dataset",
        values="Correlation"
    )
)

feature_corr_comparison["Test - Train"] = (
    feature_corr_comparison["Test (Batch 2)"]
    - feature_corr_comparison["Train (Batch 1)"]
)

display(Markdown("### Feature ↔ Feature Correlation Shift"))
display(feature_corr_comparison)

### Correct Ridge Coefficient Mapping

,Ridge Coef
log_dQ_var,-0.066
dQ_kurt,-0.003
chargetime_med,0.012


### Feature ↔ Feature Correlation Shift

Dataset,Test (Batch 2),Train (Batch 1),Valid (Batch 1),Test - Train
Feature Pair,,,,
dQ_kurt ↔ chargetime_med,-0.557,0.360,0.168,-0.918
log_dQ_var ↔ chargetime_med,0.946,-0.812,-0.809,1.759
log_dQ_var ↔ dQ_kurt,-0.496,-0.393,-0.326,-0.104


In [33]:
# Train / Valid / Test 각각에서
# y = b0 + b1*x1 + b2*x2 + b3*x3
# 형태의 보조 OLS를 적합하여 조건부 기울기를 비교한다.
#
# 주의:
# Batch shift 진단 목적으로만 사용한다.

def diagnostic_ols_coefficients(
    scaled_df,
    table,
    dataset_name
):
    X = scaled_df[selected_features].to_numpy()
    y = table["log_cycle_life"].to_numpy()

    # intercept column 추가
    X_design = np.column_stack([
        np.ones(len(X)),
        X
    ])

    beta, _, _, _ = np.linalg.lstsq(
        X_design,
        y,
        rcond=None
    )

    y_pred = X_design @ beta
    residual = y - y_pred

    return {
        "Dataset": dataset_name,
        "Intercept": beta[0],
        **{
            feature: coef
            for feature, coef
            in zip(selected_features, beta[1:])
        },
        "Residual Std": residual.std(),
        "R2": 1 - (
            np.sum(residual ** 2)
            / np.sum((y - y.mean()) ** 2)
        )
    }


ols_diagnostic = pd.DataFrame([
    diagnostic_ols_coefficients(
        train_scaled,
        train_table,
        "Train (Batch 1)"
    ),
    diagnostic_ols_coefficients(
        valid_scaled,
        valid_table,
        "Valid (Batch 1)"
    ),
    diagnostic_ols_coefficients(
        test_scaled,
        test_table,
        "Test (Batch 2)"
    ),
]).set_index("Dataset")


display(Markdown(
    "### Diagnostic OLS: Conditional Feature Slopes"
))
display(ols_diagnostic.transpose())


display(Markdown(
    "### Frozen Ridge Coefficients"
))

display(
    ridge_coef_map.to_frame()
)

### Diagnostic OLS: Conditional Feature Slopes

Dataset,Train (Batch 1),Valid (Batch 1),Test (Batch 2)
Intercept,2.910,2.921,1.971
log_dQ_var,-0.070,-0.082,-0.052
dQ_kurt,-0.004,0.034,0.044
chargetime_med,0.009,0.027,-1.075
Residual Std,0.048,0.029,0.041
R2,0.720,0.920,0.914


### Frozen Ridge Coefficients

,Ridge Coef
log_dQ_var,-0.066
dQ_kurt,-0.003
chargetime_med,0.012


In [34]:
# Train / Valid / Test에서 다중공선성 정도를 비교
# VIF가 크면 OLS coefficient 자체를 직접 해석하기 어렵다.

from sklearn.linear_model import LinearRegression


def calculate_vif(scaled_df, dataset_name):
    X = scaled_df[selected_features].copy()

    records = []

    for target_feature in selected_features:
        other_features = [
            f for f in selected_features
            if f != target_feature
        ]

        X_other = X[other_features]
        y_feature = X[target_feature]

        aux_model = LinearRegression()
        aux_model.fit(X_other, y_feature)

        r2 = aux_model.score(X_other, y_feature)

        vif = (
            1 / (1 - r2)
            if r2 < 1
            else np.inf
        )

        records.append({
            "Dataset": dataset_name,
            "Feature": target_feature,
            "Auxiliary R2": r2,
            "VIF": vif,
        })

    return pd.DataFrame(records)


train_vif = calculate_vif(
    train_scaled,
    "Train (Batch 1)"
)

valid_vif = calculate_vif(
    valid_scaled,
    "Valid (Batch 1)"
)

test_vif = calculate_vif(
    test_scaled,
    "Test (Batch 2)"
)


vif_comparison = (
    pd.concat(
        [train_vif, valid_vif, test_vif],
        ignore_index=True
    )
    .pivot(
        index="Feature",
        columns="Dataset",
        values="VIF"
    )
)

display(Markdown("### VIF: Multicollinearity Comparison"))
display(vif_comparison)


# 참고용: 각 dataset의 최대 VIF
print("Max VIF")
print(
    vif_comparison.max(axis=0)
)

### VIF: Multicollinearity Comparison

Dataset,Test (Batch 2),Train (Batch 1),Valid (Batch 1)
Feature,,,
chargetime_med,10.609,2.958,2.977
dQ_kurt,1.470,1.189,1.153
log_dQ_var,9.703,3.044,3.237


Max VIF
Dataset
Test (Batch 2)    10.609
Train (Batch 1)    3.044
Valid (Batch 1)    3.237
dtype: float64


In [35]:
# Feature ablation diagnostic
# 모든 모델은 Batch 1 Train에서만 학습한다.
# 기존 Ridge와 동일한 alpha를 사용한다.
# Batch 2 성능은 모델 선택이 아니라 원인 진단 용도로만 확인한다.

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_percentage_error

ridge_alpha = ridge_model.alpha

feature_sets = {
    "Full (3 features)": selected_features,

    "Without chargetime_med": [
        "log_dQ_var",
        "dQ_kurt",
    ],

    "Without log_dQ_var": [
        "dQ_kurt",
        "chargetime_med",
    ],

    "Without dQ_kurt": [
        "log_dQ_var",
        "chargetime_med",
    ],
}


def cycle_mape_from_log(y_log, pred_log):
    y_cycle = 10.0 ** np.asarray(y_log)
    pred_cycle = 10.0 ** np.asarray(pred_log)

    return (
        mean_absolute_percentage_error(
            y_cycle,
            pred_cycle
        )
        * 100
    )


ablation_records = []

y_train_log = train_table["log_cycle_life"].to_numpy()
y_valid_log = valid_table["log_cycle_life"].to_numpy()
y_test_log = test_table["log_cycle_life"].to_numpy()


for name, features in feature_sets.items():

    diagnostic_model = Ridge(
        alpha=ridge_alpha
    )

    diagnostic_model.fit(
        train_scaled[features],
        y_train_log
    )

    valid_pred = diagnostic_model.predict(
        valid_scaled[features]
    )

    test_pred = diagnostic_model.predict(
        test_scaled[features]
    )

    ablation_records.append({
        "Feature Set": name,
        "Valid MAPE (%)": cycle_mape_from_log(
            y_valid_log,
            valid_pred
        ),
        "Test MAPE (%)": cycle_mape_from_log(
            y_test_log,
            test_pred
        ),
        "Gap Test - Valid (%p)": (
            cycle_mape_from_log(
                y_test_log,
                test_pred
            )
            -
            cycle_mape_from_log(
                y_valid_log,
                valid_pred
            )
        ),
    })


ablation_df = (
    pd.DataFrame(ablation_records)
    .set_index("Feature Set")
)

display(Markdown(
    "### Diagnostic Feature Ablation — Batch 1 Training Only"
))
display(ablation_df)

print(f"Fixed Ridge alpha: {ridge_alpha}")

### Diagnostic Feature Ablation — Batch 1 Training Only

,Valid MAPE (%),Test MAPE (%),Gap Test - Valid (%p)
Feature Set,,,
Full (3 features),8.344,37.345,29.000
Without chargetime_med,7.752,36.380,28.628
Without log_dQ_var,14.653,53.466,38.813
Without dQ_kurt,8.424,37.559,29.135


Fixed Ridge alpha: 1.0
